Files originally from https://github.com/LucasSilvaFerreira/Perturb_Loader

In [1]:
import mudata as md
import anndata as ad
import numpy as np
import pandas as pd
import perturbvi
import os
import pyro

smoke_test = True


In [ ]:
def load_adata(data_dir = "."):
    rna_adata = ad.read_h5ad(f"{data_dir}/ann_exp.h5ad")
    rna_adata.obs['library_size']=rna_adata.X.sum(axis=1)
    rna_adata.varm['gene_tested'] = ad.read_h5ad(f"{data_dir}/ann_Element_x_tested_genes.h5ad").to_df().T
    grna_adata = ad.read_h5ad(f"{data_dir}/ann_guide.h5ad")
    grna_adata.varm['gene_targeted'] = ad.read_h5ad(f"{data_dir}/ann_Element_guide.h5ad").to_df().T
    mdata = md.MuData({'rna':rna_adata, 'grna':grna_adata})
    mdata.write_h5mu(f'{data_dir}/gasperini_pilot_highMOI.h5mu')
    return mdata

force = True
mudata_file = "gasperini_pilot_highMOI.h5mu"
data_dir = "../../../../Data/gasperini_pilot"
if mudata_file not in os.listdir(data_dir) or force:
    mdata = load_adata(data_dir)
else:
    mdata = md.read_h5mu(os.path.join(data_dir, mudata_file))

In [ ]:
mdata['grna']

AnnData object with n_obs × n_vars = 47964 × 3115
    obs: 'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
    varm: 'gene_targeted'

In [ ]:
guides = mdata['grna'].var_names
selected_guides = list(guides[guides.str.contains(r"TSS|random|scrambled", regex=True)])
if smoke_test:
    selected_guides = selected_guides[:20] + selected_guides[-10:]
    print(selected_guides)

grna_subset = mdata['grna'][:,selected_guides]
genes = {e.split('_')[0] for e in grna_subset.var_names if '_TSS' in e}
subset_genes = [g for g in genes if g in mdata['rna'].var_names]

['ACTB_TSS|1', 'ACTB_TSS|2', 'ACTG1_TSS|1', 'ACTG1_TSS|2', 'ACYP1_TSS|1', 'ACYP1_TSS|2', 'ADIPOR1_TSS|1', 'ADIPOR1_TSS|2', 'ALDH1A2_TSS|1', 'ALDH1A2_TSS|2', 'APEX1_TSS|1', 'APEX1_TSS|2', 'APLP2_TSS|1', 'APLP2_TSS|2', 'ARF6_TSS|1', 'ARF6_TSS|2', 'ARID1A_TSS|1', 'ARID1A_TSS|2', 'ARL4A_TSS|1', 'ARL4A_TSS|2', 'scrambled_5|1', 'scrambled_5|2', 'scrambled_6|1', 'scrambled_6|2', 'scrambled_7|1', 'scrambled_7|2', 'scrambled_8|1', 'scrambled_8|2', 'scrambled_9|1', 'scrambled_9|2']


In [ ]:
# subset down to only control perturbed genes
# rna_subset = mdata['rna'][:,mdata['rna'].varm['gene_tested'].sum(axis=1).values > 0]
rna_subset =  mdata['rna'][:,subset_genes]
mdata_subset = md.MuData({'rna':rna_subset.copy(), 'grna': grna_subset.copy()})
mdata_subset

MuData object with n_obs × n_vars = 47964 × 40
  2 modalities
    rna:	47964 x 10
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count', 'library_size'
      varm:	'gene_tested'
    grna:	47964 x 30
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
      varm:	'gene_targeted'

In [ ]:
perturbvi.PERTURBVI.setup_mudata(
    mdata_subset,
    batch_key="bath_number",
    size_factor_key="library_size",
    modalities={
        "rna_layer": 'rna',
        "perturbation_layer": 'grna',
    },
)

model = perturbvi.PERTURBVI(mdata_subset)
model.view_anndata_setup()

Anndata setup with scvi-tools version 0.20.1.

Setup via `PERTURBVI.setup_anndata` with arguments:

{
│   'rna_layer': None,
│   'batch_key': 'bath_number',
│   'perturbation_layer': None,
│   'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'},
│   'size_factor_key': 'library_size'
}

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│     n_batch      │   6   │
│     n_cells      │ 47964 │
│ n_perturbations  │  30   │
│      n_vars      │  10   │
└──────────────────┴───────┘

                       Data Registry                        
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Registry Key    ┃         scvi-tools Location          ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         X         │          adata.mod['rna'].X          │
│       batch       │ adata.mod['rna'].obs['_scvi_batch']  │
│       ind_x       │    adata.mod['rna'].obs['_ind_x']    │
│ observed_lib_size │ adata.mod['rna'].obs['library_size'] │
│   perturbations   │         adata.mod['grna'].X          │
└───────────────────┴──────────────────────────────────────┘

                     batch State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃     Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['bath_number'] │     1      │          0          │
│                          │     2      │          1          │
│                          │     3      │          2          │
│                          │     4      │          3          │
│                          │     5      │          4          │
│                          │     6      │          5          │
└──────────────────────────┴────────────┴─────────────────────┘

In [ ]:
# optimizer = pyro.optim.ClippedAdam({'lr':0.001, 'lrd':0.9})
model.train(
    max_epochs=20,
    train_size=1,
    batch_size=1024,
    lr=0.1
    # plan_kwargs={"optim": optimizer},
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 1/20:   0%|          | 0/20 [00:00<?, ?it/s]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


Epoch 2/20:   5%|▌         | 1/20 [00:00<00:05,  3.21it/s, v_num=1, elbo_train=1.79e+6]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


...

Epoch 20/20:  95%|█████████▌| 19/20 [00:05<00:00,  3.33it/s, v_num=1, elbo_train=9.8e+5]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


Epoch 20/20: 100%|██████████| 20/20 [00:06<00:00,  3.29it/s, v_num=1, elbo_train=9.8e+5]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 20/20: 100%|██████████| 20/20 [00:06<00:00,  3.33it/s, v_num=1, elbo_train=9.8e+5]


In [ ]:
# optimizer = pyro.optim.ClippedAdam({'lr':0.001, 'lrd':0.9})
model.train(
    max_epochs=50,
    train_size=1,
    batch_size=1024,
    lr=0.001
    # plan_kwargs={"optim": optimizer},
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 1/50:   0%|          | 0/50 [00:00<?, ?it/s]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


Epoch 2/50:   2%|▏         | 1/50 [00:00<00:14,  3.47it/s, v_num=1, elbo_train=9.78e+5]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


...

Epoch 50/50:  98%|█████████▊| 49/50 [00:14<00:00,  3.41it/s, v_num=1, elbo_train=9.75e+5]

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pyro/util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'perturb_disp_lfc'}
  warnings.warn(


Epoch 50/50: 100%|██████████| 50/50 [00:14<00:00,  3.42it/s, v_num=1, elbo_train=9.75e+5]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [00:14<00:00,  3.36it/s, v_num=1, elbo_train=9.75e+5]


In [ ]:
%load_ext autoreload
%autoreload 2
from scipy.stats import norm

lfc_threshold = 0.1 # ~ 10% knockdown

if smoke_test:
    for gene_index in range(len(subset_genes)):
        print(mdata_subset['rna'].var_names[gene_index])

        perturb_mean_lfc_mu = pyro.get_param_store()['perturb_mean_lfc.mu'].detach().cpu().numpy()
        perturb_disp_lfc_mu = pyro.get_param_store()['perturb_disp_lfc.mu'].detach().cpu().numpy()
        lfc_cov_tril = pyro.get_param_store()['perturb_lfc.scale_tril'] * pyro.get_param_store()['scale_factor'].exp()
        lfc_cov = lfc_cov_tril @ lfc_cov_tril.transpose(dim0=-1, dim1=-2)
        perturb_mean_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        perturb_disp_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        assert perturb_mean_lfc_mu.shape == perturb_mean_lfc_sigma.shape
        perturb_z_scores = perturb_mean_lfc_mu/perturb_mean_lfc_sigma
        perturb_p_vals = norm.cdf(lfc_threshold, loc=-perturb_mean_lfc_mu, scale=perturb_mean_lfc_sigma)

        # perturb_z_scores[:,gene_index].detach().cpu().numpy()
        z_df = pd.DataFrame({'z_score': perturb_z_scores[:,gene_index],
                            'mean_mu': perturb_mean_lfc_mu[:,gene_index],
                            'disp_mu': perturb_disp_lfc_mu[:,gene_index],
                            'mu_p_val': perturb_p_vals[:,gene_index],
                            'grna': mdata_subset['grna'].var_names,})
        # z_df.hist('mu_p_val', bins=100)
        # z_df.plot(x='mean_mu',y='disp_mu', style='o')
        print(z_df.sort_values('mu_p_val', ascending=True).reset_index(drop=True).head(20))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ARID1A
      z_score   mean_mu  disp_mu      mu_p_val           grna
0  -11.565962 -0.172590      0.0  5.735927e-07   ARID1A_TSS|2
1   -0.513010 -0.007655      0.0  1.000000e+00  ALDH1A2_TSS|2
2   -0.409600 -0.006112      0.0  1.000000e+00     ACTB_TSS|1
3   -0.342810 -0.005115      0.0  1.000000e+00  scrambled_5|2
4   -0.314116 -0.004687      0.0  1.000000e+00  scrambled_7|2
5   -0.302634 -0.004516      0.0  1.000000e+00    APLP2_TSS|2
6   -0.227531 -0.003395      0.0  1.000000e+00  scrambled_9|2
7   -0.207275 -0.003093      0.0  1.000000e+00  scrambled_8|1
8   -0.178909 -0.002670      0.0  1.000000e+00    APEX1_TSS|1
9   -0.158011 -0.002358      0.0  1.000000e+00  ADIPOR1_TSS|1
10  -0.116099 -0.001732      0.0  1.000000e+00   ARID1A_TSS|1
11  -0.084291 -0.001258      0.0  1.000000e+00  ADIPOR1_TSS|2
12  -0.078578 -0.001173      0.0  1.000000e+00  scrambled_6|2
13  -0.073193 -0.001092      0.0  1.0

In [ ]:
for k, v in pyro.get_param_store().items():
    print (k, v.shape)

scale_factor torch.Size([])
log_var_mean.mu torch.Size([10])
log_var_disp.mu torch.Size([10])
batch_effect.mu torch.Size([6, 1])
batch_effect.sigma torch.Size([6, 1])
log_var_mean.sigma torch.Size([10])
log_var_disp.sigma torch.Size([10])
perturb_mean_lfc.mu torch.Size([30, 10])
perturb_disp_lfc.mu torch.Size([30, 10])
perturb_lfc.scale_tril torch.Size([30, 10, 2, 2])
